In [ ]:
#Mounting drive for data
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#IMD Data
path1 = '/content/gdrive/MyDrive/GAIAData/80-20/Tmin_2010_2014.xlsx'
IMD_Nasa = pd.read_excel(path1)
IMD_Nasa

,Date,Latitude,Longitude,IMD,NASA
0,2010-01-01,9.5,78.5,20.850000,22.31
1,2010-01-02,9.5,78.5,21.180000,22.76
2,2010-01-03,9.5,78.5,20.860001,22.12
3,2010-01-04,9.5,78.5,20.760000,21.48
4,2010-01-05,9.5,78.5,19.280001,19.58
...,...,...,...,...,...
20787,2014-12-27,12.5,80.5,20.155110,25.70
20788,2014-12-28,12.5,80.5,21.936960,25.54
20789,2014-12-29,12.5,80.5,22.427230,25.33
20790,2014-12-30,12.5,80.5,21.590350,25.47


In [ ]:
df = pd.DataFrame(IMD_Nasa)
df.dtypes

Date         datetime64[ns]
Latitude            float64
Longitude           float64
IMD                 float64
NASA                float64
dtype: object

In [ ]:
results = []

In [ ]:
def adjust_imd_values(df):

    results = []  # Store the adjusted IMD values here
    for index, row in df.iterrows():
        # Extracting current row details

        lat, lon, current_imd, date = row['Latitude'], row['Longitude'], row['IMD'], row['Date']

        # Initialize target IMD to NaN (indicating no match found initially)
        target_imd = np.nan

        # Conditions based on latitude and longitude
        if 9.5 <= lat < 11.5:
          target_row = df[(df['Latitude'] == lat + 1) & (df['Date'] == date)]
        elif lat == 11.5 and 78.5 <= lon < 79.5:
          target_row = df[(df['Longitude'] == lon + 1) & (df['Date'] == date) & (df['Latitude'] == lat)]
        elif lat == 11.5 and lon == 79.5:
          target_row = df[(df['Longitude'] == lon - 1) & (df['Date'] == date) & (df['Latitude'] == lat)]

        if not target_row.empty:
            # If a matching row is found, take its IMD value for subtraction
            target_imd = target_row.iloc[0]['IMD']
        # Perform the subtraction if a matching row was found; otherwise, leave as NaN or current IMD
        result = current_imd - target_imd if not pd.isnull(target_imd) else np.nan
        results.append(result)

    # Add the results to the DataFrame as a new column
    df['error'] = results

# Apply the function to adjust IMD values
adjust_imd_values(df)
df['error'] = df['error'].abs()

# Display the modified DataFrame
df

,Date,Latitude,Longitude,IMD,NASA,error
0,2010-01-01,9.5,78.5,20.850000,22.31,0.609999
1,2010-01-02,9.5,78.5,21.180000,22.76,0.629999
2,2010-01-03,9.5,78.5,20.860001,22.12,0.660000
3,2010-01-04,9.5,78.5,20.760000,21.48,0.219999
4,2010-01-05,9.5,78.5,19.280001,19.58,0.439999
...,...,...,...,...,...,...
20787,2014-12-27,12.5,80.5,20.155110,25.70,0.758860
20788,2014-12-28,12.5,80.5,21.936960,25.54,2.540710
20789,2014-12-29,12.5,80.5,22.427230,25.33,3.030980
20790,2014-12-30,12.5,80.5,21.590350,25.47,2.194100


In [ ]:
error_column = df['error']

# Calculate error metrics
mae = np.mean(np.abs(error_column))
mse = np.mean(error_column**2)
rmse = np.sqrt(mse)
std_deviation = np.std(error_column)

# Print calculated error metrics
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Standard Deviation of Errors:", std_deviation)

Mean Absolute Error (MAE): 2.6295500852804237
Mean Squared Error (MSE): 11.004969559740596
Root Mean Squared Error (RMSE): 3.3173738950773393
Standard Deviation of Errors: 2.022482610244724
